In [3]:
!pip install kaggle tqdm scikit-learn kagglehub

In [4]:
import os

# Kaggle username
os.environ["KAGGLE_USERNAME"] = "manishjoshimj"

# Kaggle API token
os.environ["KAGGLE_KEY"] = "KGAT_1fc1ac93b6b60d033c8915d1df085ccb"

# or upload kaggle.json

# set config dir
os.environ["KAGGLE_CONFIG_DIR"] = "/content"

In [5]:
from pathlib import Path

# Base paths
BASE_DIR = Path("/content")
RAW_DIR = BASE_DIR / "datasets_raw"       # Where raw datasets are stored
PROCESSED_DIR = BASE_DIR / "dataset"      # Final dataset folder

# Classes & splits
CLASSES = ["TB", "Normal", "OtherDisease"]
SPLITS = ["train", "val", "test"]
SPLIT_RATIOS = {"train": 0.7, "val": 0.15, "test": 0.15}

# Create folders
for split in SPLITS:
    for cls in CLASSES:
        (PROCESSED_DIR / split / cls).mkdir(parents=True, exist_ok=True)

In [6]:
# -----------------------------
# Function to download Kaggle datasets
# -----------------------------
def kaggle_download(dataset, dest_folder, unzip=True):
    """
    dataset: Kaggle dataset slug, e.g., 'tawsifurrahman/covid19-radiography-database'
    dest_folder: where to save dataset
    unzip: whether to unzip after download
    """
    cmd = f"kaggle datasets download -d {dataset} -p {dest_folder}"
    if unzip:
        cmd += " --unzip"
    print(f"Downloading {dataset} ...")
    os.system(cmd)

# Download required datasets

In [7]:
# 1️. TB datasets
kaggle_download("tawsifurrahman/tuberculosis-tb-chest-xray-dataset", RAW_DIR / "kaggle_tb")


In [8]:
# 2. Pneumonia chest X-rays (Normal + Pneumonia)
kaggle_download(
    "paultimothymooney/chest-xray-pneumonia",
    RAW_DIR / "kaggle_pneumonia"
)

In [9]:
# 3. Includes Bacterial Pneumonia, Viral Pneumonia, TB, Normal, Corona
kaggle_download(
    "omkarmanohardalvi/lungs-disease-dataset-4-types",
    RAW_DIR / "kaggle_4_disease"
)

In [10]:
# 4. Download TBX11K Simplified dataset
kaggle_download(
    "vbookshelf/tbx11k-simplified",
    RAW_DIR / "tbx11k_simplified"
)

In [11]:
# -----------------------------
# 5. Download COVID-19 Radiography dataset
# -----------------------------
kaggle_download(
    "tawsifurrahman/covid19-radiography-database",
    RAW_DIR / "covid19_radiography"
)

In [12]:
# -----------------------------
# Helper function to copy & map images
# -----------------------------
import shutil
import random

def copy_and_map_images(raw_dir, class_map, processed_dir, use_split_folder=True):
    """
    raw_dir: Path to dataset
    class_map: dict mapping original classes → TB/Normal/OtherDisease
    processed_dir: destination root
    use_split_folder: True if dataset already has train/val/test folders
    """

    if use_split_folder:
        # Use existing splits
        for split in SPLITS:
            split_dir = raw_dir / split

            if not split_dir.exists():
                continue

            for orig_class, mapped_class in class_map.items():
                class_dir = split_dir / orig_class

                if not class_dir.exists():
                    continue

                for img_path in class_dir.glob("*.*"):
                    dst_path = processed_dir / split / mapped_class / img_path.name
                    shutil.copy2(img_path, dst_path)

    else:
        # Manual split
        all_files_per_class = {}

        for orig_class, mapped_class in class_map.items():
            class_dir = raw_dir / orig_class

            if not class_dir.exists():
                continue

            all_files_per_class[mapped_class] = list(class_dir.glob("*.*"))

        for mapped_class, files in all_files_per_class.items():
            random.shuffle(files)

            n_total = len(files)
            n_train = int(SPLIT_RATIOS["train"] * n_total)
            n_val = int(SPLIT_RATIOS["val"] * n_total)
            n_test = n_total - n_train - n_val

            split_files = {
                "train": files[:n_train],
                "val": files[n_train:n_train + n_val],
                "test": files[n_train + n_val:]
            }

            for split, file_list in split_files.items():
                for f in file_list:
                    dst_path = processed_dir / split / mapped_class / f.name
                    shutil.copy2(f, dst_path)

In [13]:
# -----------------------------
# 1️. Kaggle TB dataset
# -----------------------------
KAGGLE_TB_DIR = RAW_DIR / "kaggle_tb" / "TB_Chest_Radiography_Database"

TB_CLASS_MAP = {
    "Tuberculosis": "TB",
    "Normal": "Normal"
}

copy_and_map_images(
    KAGGLE_TB_DIR,
    TB_CLASS_MAP,
    PROCESSED_DIR,
    use_split_folder=False
)

In [14]:
# -----------------------------
# Print dataset summary
# -----------------------------
print("\Dataset Summary:")
for split in SPLITS:
    print(f"\n{split.upper()}:")
    for cls in CLASSES:
        path = PROCESSED_DIR / split / cls
        count = len(list(path.glob("*.*")))
        print(f"{cls}: {count} images")

\Dataset Summary:

TRAIN:
TB: 489 images
Normal: 2450 images
OtherDisease: 0 images

VAL:
TB: 105 images
Normal: 525 images
OtherDisease: 0 images

TEST:
TB: 106 images
Normal: 525 images
OtherDisease: 0 images


<>:4: SyntaxWarning: invalid escape sequence '\D'
<>:4: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_1648/3550941921.py:4: SyntaxWarning: invalid escape sequence '\D'
  print("\Dataset Summary:")


In [15]:
# -----------------------------
# 2️. Kaggle Pneumonia dataset
# -----------------------------
KAGGLE_PNEUMONIA_DIR = RAW_DIR / "kaggle_pneumonia" / "chest_xray" / "chest_xray"

PNEUMONIA_CLASS_MAP = {
    "PNEUMONIA": "OtherDisease",
    "NORMAL": "Normal"
}

copy_and_map_images(
    KAGGLE_PNEUMONIA_DIR,
    PNEUMONIA_CLASS_MAP,
    PROCESSED_DIR,
    use_split_folder=True
)

In [16]:
# -----------------------------
# Print dataset summary
# -----------------------------
print("\Dataset Summary:")
for split in SPLITS:
    print(f"\n{split.upper()}:")
    for cls in CLASSES:
        path = PROCESSED_DIR / split / cls
        count = len(list(path.glob("*.*")))
        print(f"{cls}: {count} images")

\Dataset Summary:

TRAIN:
TB: 489 images
Normal: 3792 images
OtherDisease: 3876 images

VAL:
TB: 105 images
Normal: 534 images
OtherDisease: 9 images

TEST:
TB: 106 images
Normal: 759 images
OtherDisease: 390 images


<>:4: SyntaxWarning: invalid escape sequence '\D'
<>:4: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_1648/3550941921.py:4: SyntaxWarning: invalid escape sequence '\D'
  print("\Dataset Summary:")


In [17]:
# -----------------------------
# 3️. Kaggle 4-types lung disease dataset
# -----------------------------
KAGGLE_4_DISEASE_DIR = RAW_DIR / "kaggle_4_disease" / "Lung Disease Dataset"

FOUR_TYPE_CLASS_MAP = {
    "Bacterial Pneumonia": "OtherDisease",
    "Viral Pneumonia": "OtherDisease",
    "Corona Virus Disease": "OtherDisease",
    "Normal": "Normal",
    "Tuberculosis": "TB"
}

copy_and_map_images(
    KAGGLE_4_DISEASE_DIR,
    FOUR_TYPE_CLASS_MAP,
    PROCESSED_DIR,
    use_split_folder=True
)

In [18]:
# -----------------------------
# Print dataset summary
# -----------------------------
print("\Dataset Summary:")
for split in SPLITS:
    print(f"\n{split.upper()}:")
    for cls in CLASSES:
        path = PROCESSED_DIR / split / cls
        count = len(list(path.glob("*.*")))
        print(f"{cls}: {count} images")

\Dataset Summary:

TRAIN:
TB: 1709 images
Normal: 4204 images
OtherDisease: 7235 images

VAL:
TB: 511 images
Normal: 933 images
OtherDisease: 1206 images

TEST:
TB: 514 images
Normal: 1128 images
OtherDisease: 1511 images


<>:4: SyntaxWarning: invalid escape sequence '\D'
<>:4: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_1648/3550941921.py:4: SyntaxWarning: invalid escape sequence '\D'
  print("\Dataset Summary:")


In [19]:
# -----------------------------
# 4. TBX11K training images datset
# -----------------------------
TBX11K_RAW_DIR = RAW_DIR / "tbx11k_simplified" / "tbx11k-simplified"

# Training images folder
TBX11K_TRAIN_DIR = TBX11K_RAW_DIR / "images"

# All images in images/ are TB
copy_and_map_images(
    TBX11K_TRAIN_DIR,
    {"": "TB"},          # map all images to TB
    PROCESSED_DIR,
    use_split_folder=False
)

In [20]:
# -----------------------------
# Print dataset summary
# -----------------------------
print("\Dataset Summary:")
for split in SPLITS:
    print(f"\n{split.upper()}:")
    for cls in CLASSES:
        path = PROCESSED_DIR / split / cls
        count = len(list(path.glob("*.*")))
        print(f"{cls}: {count} images")

\Dataset Summary:

TRAIN:
TB: 7589 images
Normal: 4204 images
OtherDisease: 7235 images

VAL:
TB: 1771 images
Normal: 933 images
OtherDisease: 1206 images

TEST:
TB: 1774 images
Normal: 1128 images
OtherDisease: 1511 images


<>:4: SyntaxWarning: invalid escape sequence '\D'
<>:4: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_1648/3550941921.py:4: SyntaxWarning: invalid escape sequence '\D'
  print("\Dataset Summary:")


In [21]:
# -----------------------------
# 5. COVID dataset dataset
# -----------------------------
COVID_DIR = RAW_DIR / "covid19_radiography" / "COVID-19_Radiography_Dataset"

# Map to your dataset classes
COVID_CLASS_MAP = {
    "COVID": "OtherDisease",
    "Normal": "Normal",
    "Viral Pneumonia": "OtherDisease",
    "Lung_Opacity": "OtherDisease"
}

# Copy images only, ignoring masks
for orig_class, mapped_class in COVID_CLASS_MAP.items():
    class_images_dir = COVID_DIR / orig_class / "images"  # point to images/ subfolder
    if not class_images_dir.exists():
        print(f"Warning: {class_images_dir} does not exist")
        continue

    # Map all images to processed dataset, manual split
    copy_and_map_images(
        class_images_dir,
        {"": mapped_class},       # all images in this folder → mapped_class
        PROCESSED_DIR,
        use_split_folder=False    # manually split into train/val/test
    )

In [22]:
# -----------------------------
# Print dataset summary
# -----------------------------
print("\Dataset Summary:")
for split in SPLITS:
    print(f"\n{split.upper()}:")
    for cls in CLASSES:
        path = PROCESSED_DIR / split / cls
        count = len(list(path.glob("*.*")))
        print(f"{cls}: {count} images")

<>:4: SyntaxWarning: invalid escape sequence '\D'
<>:4: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_1648/3550941921.py:4: SyntaxWarning: invalid escape sequence '\D'
  print("\Dataset Summary:")


\Dataset Summary:

TRAIN:
TB: 7589 images
Normal: 9612 images
OtherDisease: 14915 images

VAL:
TB: 1771 images
Normal: 2382 images
OtherDisease: 2850 images

TEST:
TB: 1774 images
Normal: 2580 images
OtherDisease: 3160 images


In [23]:
# -----------------------------
# 4. TBX11K training images from test datset
# -----------------------------
TBX11K_RAW_DIR = RAW_DIR / "tbx11k_simplified" / "tbx11k-simplified"

# Test images folder
TBX11K_TEST_DIR = TBX11K_RAW_DIR / "test"

# All images in test/ are TB
copy_and_map_images(
    TBX11K_TEST_DIR,
    {"": "TB"},          # map all images to TB
    PROCESSED_DIR,
    use_split_folder=False
)

In [24]:
# -----------------------------
# Print dataset summary
# -----------------------------
print("\Dataset Summary:")
for split in SPLITS:
    print(f"\n{split.upper()}:")
    for cls in CLASSES:
        path = PROCESSED_DIR / split / cls
        count = len(list(path.glob("*.*")))
        print(f"{cls}: {count} images")

<>:4: SyntaxWarning: invalid escape sequence '\D'
<>:4: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_1648/3550941921.py:4: SyntaxWarning: invalid escape sequence '\D'
  print("\Dataset Summary:")


\Dataset Summary:

TRAIN:
TB: 9900 images
Normal: 9612 images
OtherDisease: 14915 images

VAL:
TB: 2266 images
Normal: 2382 images
OtherDisease: 2850 images

TEST:
TB: 2270 images
Normal: 2580 images
OtherDisease: 3160 images


In [25]:
import os
def get_folder_size(folder_path):
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(folder_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size

dataset_folder = "/content/dataset"
size_bytes = get_folder_size(dataset_folder)
size_mb = size_bytes / (1024 * 1024)
print(f"The size of the '{dataset_folder}' folder is: {size_mb:.2f} MB")

The size of the '/content/dataset' folder is: 7800.45 MB


In [26]:
! git clone https://github.com/bibekjoshi01/TBVision.git

Cloning into 'TBVision'...
remote: Enumerating objects: 526, done.
remote: Counting objects: 100% (526/526), done.
remote: Compressing objects: 100% (378/378), done.
remote: Total 526 (delta 236), reused 411 (delta 125), pack-reused 0 (from 0)
Receiving objects: 100% (526/526), 15.32 MiB | 23.00 MiB/s, done.
Resolving deltas: 100% (236/236), done.


In [27]:
! cd TBVision && git pull

Already up to date.


In [ ]:
! cd TBVision && python -m tbvision.xraytb_net.training.train_classifier \
  --data-dir dataset \
  --save-dir weights \
  --mode ensemble \
  --backbones densenet121 efficientnet_b3 resnet50 \
  --epochs 15 \
  --batch-size 32 \
  --lr 1e-4 \
  --image-size 224

Backbones: ['densenet121', 'efficientnet_b3', 'resnet50'] (mode=ensemble)
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth
100% 30.8M/30.8M [00:00<00:00, 171MB/s]
model.safetensors: 100% 49.3M/49.3M [00:00<00:00, 60.3MB/s]
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100% 97.8M/97.8M [00:00<00:00, 139MB/s]

Loading datasets...
/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)
Train samples: 34425
Val samples: 7496
Test samples: 8010
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, whi